In [1]:
import fitz
from sentence_transformers import SentenceTransformer
import chromadb
import requests

# ---------- Ingest ----------
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = []

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")
        if text.strip():
            full_text.append(f"[Page {page_num}]\n{text}")

    doc.close()
    return "\n".join(full_text)


def chunk_text(text, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks


# ---------- Retrieve ----------
model = SentenceTransformer("all-MiniLM-L6-v2")

def get_embedding(text):
    return model.encode(text).tolist()


def create_collection():
    client = chromadb.Client()
    return client.get_or_create_collection(name="research_papers")


def add_chunks(collection, chunks):
    for i, chunk in enumerate(chunks):
        collection.add(
            documents=[chunk],
            embeddings=[get_embedding(chunk)],
            ids=[f"chunk_{i}"]
        )


def retrieve_chunks(collection, question, top_k=3):
    query_embedding = get_embedding(question)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return results["documents"][0]


# ---------- Generate ----------
import subprocess

def generate_answer_local(question, context, model_name="llama3.2"):
    prompt = f"""
You are a research paper assistant.
Answer the question using only the context below.
If the answer is not in the context, say:
"I could not find the answer in the provided paper."

Context:
{context}

Question:
{question}
"""
    result = subprocess.run(
        ["C:\\Users\\kruta\\AppData\\Local\\Programs\\Ollama\\ollama.exe", "run", model_name],
        input=prompt.encode(),
        stdout=subprocess.PIPE
    )
    return result.stdout.decode()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
pdf_path = r"C:\Users\kruta\Desktop\ML PREP\RAG\data\paperNN.pdf"

# Step 1: Extract text
text = extract_text_from_pdf(pdf_path)

# Step 2: Chunk text
chunks = chunk_text(text)

print("Text extracted.")
print("Total chunks:", len(chunks))

# Step 3: Create collection
collection = create_collection()

# Optional: avoid duplicate-add issue by recreating with a fresh name if needed
# For now, only run add_chunks once per fresh kernel session
add_chunks(collection, chunks)

print("Chunks added to collection.")
print("Collection count:", collection.count())

# Step 4: Ask question
question = input("Ask a question: ")

# Step 5: Retrieve relevant chunks
retrieved_chunks = retrieve_chunks(collection, question, top_k=3)

print("\nRetrieved Chunks:\n")
for i, chunk in enumerate(retrieved_chunks, start=1):
    print(f"Chunk {i}:\n{chunk[:500]}")
    print("\n---\n")

# Step 6: Generate final answer
answer = generate_answer_local(question, retrieved_chunks)

print("\nFinal Answer:\n")
print(answer)

Text extracted.
Total chunks: 36
Chunks added to collection.
Collection count: 36


Ask a question:  what is the main goal of the paper?



Retrieved Chunks:

Chunk 1:
Erban, From microscopic to macroscopic descriptions of cell migration on growing domains, Bulletin of Mathematical Biology, 72 (2010), pp. 719–762. [5] N. Barkai and S. Leibler, Robustness in simple biochemical networks, Nature, 387 (1997), pp. 913–917. [6] N. Bellomo, A. Bellouquid, Y. Tao, and M. Winkler, Toward a mathematical theory of Keller-Segel models of pattern formation in biological tissues, Mathematical Models and Methods in Applied Sciences, 25 (2015), pp. 1663–1763. [7] J. Dallon an

---

Chunk 2:
[11]. We again assume that each cell moves along the x-axis at a constant speed β > 0, i.e. its velocity Vi(t) only takes one of two possible values, Vi(t) = ±β. Each cell reverses its direction at random instants of time according to the Poisson process with turning frequency (5.3) ReLU λ0 + λ0 (1 + 2ta)  Yi −ψ(S)  ta β2 ! , where λ0 > 0, the function ReLU is deﬁned by (2.9) and signal S is evaluated at the current position of the cell, Xi(t). Since

In [ ]:
# Q1

# What is the main goal of the paper?

# ✅ Answer:
# The paper aims to estimate macroscopic chemotactic sensitivity from microscopic (individual-based) models using neural networks.

# Q2

# What equation describes the macroscopic chemotaxis behavior?

# ✅ Answer:
# A drift-diffusion PDE (chemotaxis equation):

# ∂c∂t=∇⋅(D∇c−cχ(S)∇S)
# ∂t
# ∂c
# 	​

# =∇⋅(D∇c−cχ(S)∇S)

In [ ]:
import streamlit as st
import fitz
from sentence_transformers import SentenceTransformer
import chromadb
import subprocess

# ---------- Helper functions ----------

def extract_text_from_pdf(pdf_file):
    """Extract text from PDF using PyMuPDF."""
    doc = fitz.open(stream=pdf_file.read(), filetype="pdf")
    full_text = []

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")
        if text.strip():
            full_text.append(f"[Page {page_num}]\n{text}")

    doc.close()
    return "\n".join(full_text)

def chunk_text(text, chunk_size=300, overlap=50):
    """Split text into overlapping chunks."""
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

# ---------- Initialize embeddings & ChromaDB collection ----------

model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
collection_name = "research_papers"
collection = client.get_or_create_collection(name=collection_name)

def get_embedding(text):
    """Return embedding as a list for ChromaDB."""
    return model.encode(text).tolist()

def add_chunks_to_collection(chunks):
    """Add text chunks to ChromaDB with proper metadata."""
    if not chunks:
        st.warning("No text chunks to insert into the collection!")
        return

    ids = [str(i) for i in range(len(chunks))]
    documents = chunks
    # Make sure metadata is non-empty for every chunk
    metadatas = [{"source": "uploaded_pdf", "chunk_index": i} for i in range(len(chunks))]

    # Debug: print first 3 chunks
    for i in range(min(3, len(chunks))):
        print(f"Chunk {i}: {chunks[i][:50]}...")
        print(f"Metadata: {metadatas[i]}")

    collection.upsert(
        ids=ids,
        documents=documents,
        metadatas=metadatas
    )
    st.success(f"Inserted {len(chunks)} chunks into collection with metadata.")

def retrieve_chunks(question, top_k=3):
    """Retrieve relevant chunks from ChromaDB using semantic search."""
    query_embedding = get_embedding(question)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return results["documents"][0]  # Returns a list of chunks

def generate_answer_local(question, retrieved_chunks, model_path=r"C:\Users\kruta\AppData\Local\Programs\Ollama\ollama.exe", model_name="llama3.2"):
    """Generate answer from local LLM using subprocess."""
    context = "\n\n".join(retrieved_chunks)
    prompt = f"""
You are a research paper assistant.
Answer the question using only the context below.
If the answer is not in the context, say:
"I could not find the answer in the provided paper."

Context:
{context}

Question:
{question}
"""
    result = subprocess.run(
        [model_path, "run", model_name],
        input=prompt.encode(),
        stdout=subprocess.PIPE
    )
    return result.stdout.decode()

# ---------- Streamlit UI ----------

st.title("📄 Research Paper Q&A")

uploaded_file = st.file_uploader("Upload a PDF", type=["pdf"])

if uploaded_file:
    text = extract_text_from_pdf(uploaded_file)
    if text.strip():  # Ensure PDF has readable text
        chunks = chunk_text(text)
        add_chunks_to_collection(chunks)

        question = st.text_input("Ask a question about the paper:")

        if question:
            retrieved_chunks = retrieve_chunks(question)
            answer = generate_answer_local(question, retrieved_chunks)
            st.subheader("Answer:")
            st.write(answer)
    else:
        st.error("The uploaded PDF has no readable text!")